# Agent E2E Testing

> End-to-end testing framework for agent MCP usage patterns

In [ ]:
#| default_exp tests.agent_e2e

In [ ]:
#| hide
from nbdev.showdoc import *

## Imports

In [ ]:
#| export
from __future__ import annotations
from typing import Any, Dict, List, Optional, Callable
from dataclasses import dataclass, field
from datetime import datetime
from pathlib import Path
import json
import hashlib
import time

## Tool Call Recording

Record and analyze MCP tool calls made during a session.

In [ ]:
#| export
@dataclass
class ToolCall:
    """Record of a single MCP tool call."""
    tool_name: str
    args: Dict[str, Any]
    result: Optional[Dict[str, Any]] = None
    timestamp: float = field(default_factory=time.time)
    duration_ms: float = 0.0
    success: bool = True
    error: Optional[str] = None
    
    @property
    def args_hash(self) -> str:
        """Hash of args for deduplication detection."""
        return hashlib.md5(json.dumps(self.args, sort_keys=True).encode()).hexdigest()[:8]
    
    def __repr__(self):
        status = "✓" if self.success else "✗"
        return f"{status} {self.tool_name}({self.args_hash}) [{self.duration_ms:.0f}ms]"

In [ ]:
#| export
@dataclass
class ToolCallSession:
    """Collection of tool calls in a session with analysis capabilities."""
    calls: List[ToolCall] = field(default_factory=list)
    session_id: str = field(default_factory=lambda: datetime.now().strftime("%Y%m%d_%H%M%S"))
    
    def add(self, call: ToolCall) -> None:
        """Add a tool call to the session."""
        self.calls.append(call)
    
    def find_duplicates(self) -> List[tuple[ToolCall, ToolCall]]:
        """Find duplicate tool calls (same tool + args)."""
        seen: Dict[str, ToolCall] = {}
        duplicates = []
        for call in self.calls:
            key = f"{call.tool_name}:{call.args_hash}"
            if key in seen:
                duplicates.append((seen[key], call))
            else:
                seen[key] = call
        return duplicates
    
    def get_tool_counts(self) -> Dict[str, int]:
        """Count calls per tool."""
        counts: Dict[str, int] = {}
        for call in self.calls:
            counts[call.tool_name] = counts.get(call.tool_name, 0) + 1
        return counts
    
    def get_sequence(self) -> List[str]:
        """Get sequence of tool names in order."""
        return [call.tool_name for call in self.calls]
    
    def contains_pattern(self, pattern: List[str]) -> bool:
        """Check if the session contains a specific sequence pattern."""
        seq = self.get_sequence()
        for i in range(len(seq) - len(pattern) + 1):
            if seq[i:i+len(pattern)] == pattern:
                return True
        return False
    
    def to_dict(self) -> Dict[str, Any]:
        """Serialize session to dict."""
        return {
            "session_id": self.session_id,
            "calls": [
                {
                    "tool_name": c.tool_name,
                    "args": c.args,
                    "timestamp": c.timestamp,
                    "duration_ms": c.duration_ms,
                    "success": c.success,
                    "error": c.error
                }
                for c in self.calls
            ],
            "stats": {
                "total_calls": len(self.calls),
                "unique_tools": len(set(c.tool_name for c in self.calls)),
                "duplicates": len(self.find_duplicates()),
                "success_rate": sum(1 for c in self.calls if c.success) / max(len(self.calls), 1)
            }
        }
    
    def save(self, path: Path) -> None:
        """Save session to JSON file."""
        path.write_text(json.dumps(self.to_dict(), indent=2))
    
    @classmethod
    def load(cls, path: Path) -> 'ToolCallSession':
        """Load session from JSON file."""
        data = json.loads(path.read_text())
        session = cls(session_id=data["session_id"])
        for c in data["calls"]:
            session.add(ToolCall(
                tool_name=c["tool_name"],
                args=c["args"],
                timestamp=c["timestamp"],
                duration_ms=c["duration_ms"],
                success=c["success"],
                error=c.get("error")
            ))
        return session

## Test Scenarios

Define expected behaviors for different agent tasks.

In [ ]:
#| export
@dataclass
class TestScenario:
    """Define expected agent behavior for a task."""
    name: str
    description: str
    prompt: str  # The task prompt to give the agent
    
    # Expected behaviors
    required_tools: List[str] = field(default_factory=list)  # Must call these
    forbidden_tools: List[str] = field(default_factory=list)  # Must NOT call
    required_patterns: List[List[str]] = field(default_factory=list)  # Must contain sequences
    max_duplicates: int = 0  # Maximum allowed duplicate calls
    
    def validate(self, session: ToolCallSession) -> Dict[str, Any]:
        """Validate a session against this scenario."""
        results = {
            "scenario": self.name,
            "passed": True,
            "failures": []
        }
        
        called_tools = set(c.tool_name for c in session.calls)
        
        # Check required tools
        for tool in self.required_tools:
            if tool not in called_tools:
                results["passed"] = False
                results["failures"].append(f"Missing required tool: {tool}")
        
        # Check forbidden tools
        for tool in self.forbidden_tools:
            if tool in called_tools:
                results["passed"] = False
                results["failures"].append(f"Used forbidden tool: {tool}")
        
        # Check patterns
        for pattern in self.required_patterns:
            if not session.contains_pattern(pattern):
                results["passed"] = False
                results["failures"].append(f"Missing pattern: {' -> '.join(pattern)}")
        
        # Check duplicates
        duplicates = session.find_duplicates()
        if len(duplicates) > self.max_duplicates:
            results["passed"] = False
            results["failures"].append(
                f"Too many duplicates: {len(duplicates)} (max {self.max_duplicates})"
            )
        
        results["stats"] = {
            "total_calls": len(session.calls),
            "unique_tools": len(called_tools),
            "duplicates": len(duplicates)
        }
        
        return results

## Built-in Scenarios

In [ ]:
#| export
EDIT_PY_FILE_SCENARIO = TestScenario(
    name="edit_py_file",
    description="Agent should check if file is generated before editing",
    prompt="Edit the function `foo` in nbdev_mcp/utils/paths.py to add logging",
    required_tools=["check_if_generated"],
    required_patterns=[["check_if_generated"]],  # Must check first
    max_duplicates=0
)

FIX_LINT_SCENARIO = TestScenario(
    name="fix_lint_errors",
    description="Agent should lint, fix, then verify",
    prompt="Fix all lint errors in this nbdev project",
    required_tools=["lint_rules"],
    max_duplicates=1  # One re-check allowed
)

EXPLORE_CODEBASE_SCENARIO = TestScenario(
    name="explore_codebase",
    description="Agent should use analysis tools efficiently",
    prompt="What modules does this project export?",
    required_tools=["current_project"],  # Should establish context
    forbidden_tools=["edit_notebook_cell"],  # Should not edit during exploration
    max_duplicates=0
)

FIND_DUPLICATES_SCENARIO = TestScenario(
    name="find_duplicates",
    description="Agent should use deduplication tools",
    prompt="Find and consolidate duplicate code in this project",
    required_tools=["find_duplicates"],
    max_duplicates=0
)

RUN_TESTS_SCENARIO = TestScenario(
    name="run_tests",
    description="Agent should run tests appropriately",
    prompt="Run the tests for this project and fix any failures",
    required_tools=["nbdev_test"],  # or pytest_run
    max_duplicates=1
)

SCENARIOS = {
    "edit_py_file": EDIT_PY_FILE_SCENARIO,
    "fix_lint": FIX_LINT_SCENARIO,
    "explore": EXPLORE_CODEBASE_SCENARIO,
    "find_duplicates": FIND_DUPLICATES_SCENARIO,
    "run_tests": RUN_TESTS_SCENARIO,
}

## MCP Recorder Middleware

Wrap MCP tool calls to record them for analysis.

In [ ]:
#| export
class McpRecorder:
    """Middleware to record MCP tool calls."""
    
    def __init__(self, session: Optional[ToolCallSession] = None):
        self.session = session or ToolCallSession()
        self._original_handlers: Dict[str, Callable] = {}
    
    def wrap_tool(self, tool_name: str, handler: Callable) -> Callable:
        """Wrap a tool handler to record calls."""
        self._original_handlers[tool_name] = handler
        
        async def recorded_handler(**kwargs):
            start = time.time()
            call = ToolCall(tool_name=tool_name, args=kwargs)
            
            try:
                result = await handler(**kwargs)
                call.result = result
                call.success = result.get("ok", True) if isinstance(result, dict) else True
            except Exception as e:
                call.success = False
                call.error = str(e)
                raise
            finally:
                call.duration_ms = (time.time() - start) * 1000
                self.session.add(call)
            
            return result
        
        return recorded_handler
    
    def get_report(self) -> str:
        """Generate a human-readable report."""
        lines = [f"Session: {self.session.session_id}"]
        lines.append(f"Total calls: {len(self.session.calls)}")
        lines.append("")
        
        lines.append("Tool counts:")
        for tool, count in sorted(self.session.get_tool_counts().items()):
            lines.append(f"  {tool}: {count}")
        
        duplicates = self.session.find_duplicates()
        if duplicates:
            lines.append("")
            lines.append(f"Duplicates ({len(duplicates)}):")
            for orig, dup in duplicates:
                lines.append(f"  {orig.tool_name}({orig.args_hash})")
        
        lines.append("")
        lines.append("Call sequence:")
        for i, call in enumerate(self.session.calls):
            lines.append(f"  {i+1}. {call}")
        
        return "\n".join(lines)

## Test Runner

In [ ]:
#| export
def run_scenario_validation(
    session_path: Path,
    scenario_name: str
) -> Dict[str, Any]:
    """Load a recorded session and validate against a scenario."""
    session = ToolCallSession.load(session_path)
    scenario = SCENARIOS.get(scenario_name)
    
    if not scenario:
        return {"error": f"Unknown scenario: {scenario_name}"}
    
    return scenario.validate(session)

In [ ]:
#| export
def analyze_session_efficiency(session: ToolCallSession) -> Dict[str, Any]:
    """Analyze session for efficiency metrics."""
    duplicates = session.find_duplicates()
    total_time = sum(c.duration_ms for c in session.calls)
    
    return {
        "total_calls": len(session.calls),
        "unique_calls": len(session.calls) - len(duplicates),
        "duplicate_calls": len(duplicates),
        "duplicate_ratio": len(duplicates) / max(len(session.calls), 1),
        "total_time_ms": total_time,
        "avg_call_time_ms": total_time / max(len(session.calls), 1),
        "wasted_time_ms": sum(
            dup.duration_ms for _, dup in duplicates
        ),
        "efficiency_score": 1.0 - (len(duplicates) / max(len(session.calls), 1))
    }

In [ ]:
#| export
from nbdev_mcp.utils.logs import log

# Global recorder instance
_recorder: Optional[McpRecorder] = None
_recording_enabled: bool = False


def get_recorder() -> Optional[McpRecorder]:
    """Get the current recorder instance."""
    global _recorder
    return _recorder


def start_recording() -> McpRecorder:
    """Start recording tool calls."""
    global _recorder, _recording_enabled
    _recorder = McpRecorder()
    _recording_enabled = True
    log.info(f"Started recording session: {_recorder.session.session_id}")
    return _recorder


def stop_recording() -> Optional[ToolCallSession]:
    """Stop recording and return the session."""
    global _recorder, _recording_enabled
    _recording_enabled = False
    if _recorder:
        session = _recorder.session
        log.info(f"Stopped recording. Total calls: {len(session.calls)}")
        return session
    return None


def is_recording() -> bool:
    """Check if recording is enabled."""
    return _recording_enabled


def record_call(tool_name: str, args: Dict[str, Any], result: Any, duration_ms: float, success: bool, error: Optional[str] = None) -> None:
    """Record a tool call if recording is enabled."""
    global _recorder
    if _recording_enabled and _recorder:
        call = ToolCall(
            tool_name=tool_name,
            args=args,
            result=result if isinstance(result, dict) else {"value": str(result)},
            duration_ms=duration_ms,
            success=success,
            error=error
        )
        _recorder.session.add(call)

In [ ]:
#| export
def add_recording_tools(mcp) -> None:
    """Add recording control tools to MCP server."""
    from mcp.server.fastmcp import Context
    
    @mcp.tool()
    def start_session_recording(ctx: Context) -> Dict[str, Any]:
        """Start recording MCP tool calls for analysis.
        
        Returns session ID that can be used to retrieve the recording.
        """
        recorder = start_recording()
        return {
            "ok": True,
            "session_id": recorder.session.session_id,
            "message": "Recording started"
        }
    
    @mcp.tool()
    def stop_session_recording(
        ctx: Context,
        save_path: Optional[str] = None
    ) -> Dict[str, Any]:
        """Stop recording and optionally save to file.
        
        Args:
            save_path: Optional path to save the session JSON.
        
        Returns session statistics and optionally the file path.
        """
        session = stop_recording()
        if not session:
            return {"ok": False, "error": "No active recording"}
        
        result = {
            "ok": True,
            "session_id": session.session_id,
            "stats": session.to_dict()["stats"]
        }
        
        if save_path:
            path = Path(save_path)
            session.save(path)
            result["saved_to"] = str(path)
        
        return result
    
    @mcp.tool()
    def get_recording_status(ctx: Context) -> Dict[str, Any]:
        """Check if recording is active and get current stats."""
        recorder = get_recorder()
        if not is_recording() or not recorder:
            return {"ok": True, "recording": False}
        
        return {
            "ok": True,
            "recording": True,
            "session_id": recorder.session.session_id,
            "calls_so_far": len(recorder.session.calls),
            "duplicates_so_far": len(recorder.session.find_duplicates())
        }
    
    @mcp.tool()
    def validate_session(
        ctx: Context,
        session_path: str,
        scenario: str
    ) -> Dict[str, Any]:
        """Validate a recorded session against a test scenario.
        
        Args:
            session_path: Path to session JSON file.
            scenario: Name of scenario (edit_py_file, fix_lint, explore, find_duplicates, run_tests).
        """
        return run_scenario_validation(Path(session_path), scenario)

## Next

In [ ]:
#| export


## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()